# Integrated GCN + Autoencoder Model

Previously, our architecture relied on two separately trained models: a graph convolutional network (GCN) for cell type classificication using protein features and an autencoder for dimensional reduction to d=3 and recovery of the embeddings from the last layer of the GCN.

This notebook is an attempt at an integrated structure that optimizes a single network on cross-entropy loss while

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [12]:
# !pip install torch torch-geometric scikit-learn matplotlib pandas numpy opencv-python

import math, os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

from sklearn.preprocessing import LabelEncoder
from sklearn.neighbors import NearestNeighbors

import matplotlib.pyplot as plt

from torch_geometric.data import Dataset, Data
from torch_geometric.loader import DataLoader
from torch_geometric.nn import GCNConv

# Reproducibility
torch.manual_seed(42)
np.random.seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device


device(type='cuda')

## Data Preparation
Below, we construct a class which represents each individual region and builds a graph using the nearest neighbors of each cell.
Afterwards, we use this class to create objects for each of the unique regions in our dataset.

In [3]:
class RegionDataset(Dataset):
    """
    Wraps a flat dataframe into a list of region-specific graphs.
    One graph per unique 'unique_region' value, edges by k-NN on features.
    """
    def __init__(self, features_df, unique_region, labels_encoded, x_positions, y_positions, k_neighbors=5):
        super().__init__()
        self.region_list = np.unique(unique_region)
        self.features = features_df.values.astype(np.float32)
        self.unique_region = unique_region
        self.labels = labels_encoded.astype(np.int64)
        self.x_positions = x_positions.astype(np.float32)
        self.y_positions = y_positions.astype(np.float32)
        self.k = int(k_neighbors)

    def len(self):
        return len(self.region_list)

    def get(self, idx):
        region_id = self.region_list[idx]
        mask = (self.unique_region == region_id)

        X = self.features[mask]
        y = self.labels[mask]
        x_pos = self.x_positions[mask]
        y_pos = self.y_positions[mask]

        # kNN edges (directed; PyG will handle batching)
        nbrs = NearestNeighbors(n_neighbors=self.k)
        nbrs.fit(X)
        _, indices = nbrs.kneighbors(X)

        edge_list = []
        for i, neigh in enumerate(indices):
            for j in neigh:
                if i != j:
                    edge_list.append([i, j])
        edge_index = torch.tensor(edge_list, dtype=torch.long).T.contiguous()

        data = Data(
            x=torch.tensor(X, dtype=torch.float32),
            edge_index=edge_index,
            y=torch.tensor(y, dtype=torch.long),
        )
        data.x_pos = torch.tensor(x_pos)
        data.y_pos = torch.tensor(y_pos)
        data.region_id = region_id
        return data


In [4]:
# Set pandas to show all columns for debugging
pd.set_option('display.max_columns', None)

# Load the dataset
df = pd.read_csv('/content/drive/MyDrive/1024x1024_dataset.csv', index_col=0)

unique_region = df["unique_region"].values
labels = df["Cell Type"].values
x_positions = df['x'].to_numpy()
y_positions = df['y'].to_numpy()

drop_cols = [
    "Cell Type","unique_region","donor","array","Xcorr","Ycorr","Tissue_location","tissue","region",
    "OLFM4","FAP","CD25","CK7","MUC6","Cell Type em","Cell subtype","Neighborhood","Neigh_sub",
    "Neighborhood_Ind","NeighInd_sub","Community","Major Community","Tissue Segment","Tissue Unit","CollIV"
]
features_df = df.drop(columns=[c for c in drop_cols if c in df.columns])

label_encoder = LabelEncoder()
labels_encoded = label_encoder.fit_transform(labels)
num_classes = len(label_encoder.classes_)
print("Classes:", num_classes, " | Features:", features_df.shape[1])

dataset = RegionDataset(
    features_df=features_df,
    unique_region=unique_region,
    labels_encoded=labels_encoded,
    x_positions=x_positions,
    y_positions=y_positions,
    k_neighbors=5,
)
len(dataset), dataset[0]

/tmp/ipython-input-1097491494.py:5: DtypeWarning: Columns (73) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('/content/drive/MyDrive/1024x1024_dataset.csv', index_col=0)


Classes: 25  | Features: 48


(66,
 Data(x=[21016, 48], edge_index=[2, 84064], y=[21016], x_pos=[21016], y_pos=[21016], region_id='B004_Ascending'))

In [5]:
batch_size = 5  # tune
loader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

In [6]:
class P2MBlock(nn.Module):
    """
    Anneals GCN message passing into an MLP layer:
        h' = (1 - alpha)*GCN(h) + alpha*Linear(h)
    alpha is either scheduled externally (alpha_override) or kept fixed per block.
    """
    def __init__(self, dim, dropout=0.1, alpha_init=0.0, learn_alpha=False):
        super().__init__()
        self.gcn = GCNConv(dim, dim, normalize=True)
        self.ff  = nn.Linear(dim, dim)
        self.dropout = nn.Dropout(dropout)

        self.learn_alpha = learn_alpha
        if learn_alpha:
            self._alpha_raw = nn.Parameter(torch.tensor(0.0))
            with torch.no_grad():
                # inverse sigmoid to initialize near alpha_init
                self._alpha_raw.copy_(torch.log(torch.tensor(alpha_init + 1e-6)) - torch.log(torch.tensor(1 - alpha_init + 1e-6)))
        else:
            self.register_buffer('_alpha_const', torch.tensor(float(alpha_init)))

        # initialize ff near-identity
        with torch.no_grad():
            nn.init.eye_(self.ff.weight)
            nn.init.zeros_(self.ff.bias)

    def alpha(self):
        if self.learn_alpha:
            return torch.sigmoid(self._alpha_raw)
        else:
            return self._alpha_const

    def forward(self, x, edge_index, alpha_override=None):
        a = self.alpha() if alpha_override is None else torch.clamp(alpha_override, 0.0, 1.0)
        g = self.gcn(x, edge_index)
        m = self.ff(x)
        h = (1 - a) * g + a * m
        return F.relu(self.dropout(h))


In [7]:
class GCN3DBottleneckToClasses(nn.Module):
    """
    in_dim → pre → (k × P2M) → down to 3-D → up to [num_classes] logits
    Cross-entropy is taken on the final logits. Gradients flow through 3-D code.
    """
    def __init__(
        self,
        in_dim,
        num_classes=25,
        hidden_dim=128,
        num_p2m_blocks=3,
        down_dims=(64, 32, 16, 3),
        up_dims=(16, 32, 64),
        dropout=0.10,
        learn_alpha=False,
        alpha_init=0.0,
        add_aux=False
    ):
        super().__init__()
        self.num_classes = num_classes
        self.add_aux = add_aux

        self.pre = nn.Sequential(
            nn.Linear(in_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout)
        )

        self.blocks = nn.ModuleList([
            P2MBlock(hidden_dim, dropout=dropout, alpha_init=alpha_init, learn_alpha=learn_alpha)
            for _ in range(num_p2m_blocks)
        ])

        self.aux_head_pre = nn.Linear(hidden_dim, num_classes) if add_aux else None

        # down to 3
        d = hidden_dim
        dn = []
        for fd in down_dims:
            dn.append(nn.Linear(d, fd))
            if fd != 3:
                dn += [nn.ReLU(), nn.Dropout(dropout)]
            d = fd
        self.down = nn.Sequential(*dn)

        self.code_norm = nn.LayerNorm(3, elementwise_affine=True)

        # up to classes
        u = 3
        up = []
        for ud in up_dims:
            up += [nn.Linear(u, ud), nn.ReLU(), nn.Dropout(dropout)]
            u = ud
        up += [nn.Linear(u, num_classes)]
        self.up = nn.Sequential(*up)

        self.aux_head_post = nn.Linear(3, num_classes) if add_aux else None

    def forward(self, x, edge_index, alpha_override=None, return_all=False):
        h = self.pre(x)
        for b in self.blocks:
            h = b(h, edge_index, alpha_override=alpha_override)

        aux_pre = self.aux_head_pre(h) if self.add_aux else None

        z3 = self.down(h)          # [N,3]
        z3 = self.code_norm(z3)

        aux_post = self.aux_head_post(z3) if self.add_aux else None
        logits = self.up(z3)       # [N,num_classes]

        if return_all:
            return dict(h=h, z3=z3, logits=logits, aux_pre=aux_pre, aux_post=aux_post)
        return z3, logits


In [8]:
num_features = features_df.shape[1]

model = GCN3DBottleneckToClasses(
    in_dim=num_features,
    num_classes=num_classes,     # should be 25
    hidden_dim=128,
    num_p2m_blocks=3,
    down_dims=(64, 32, 16, 3),
    up_dims=(16, 32, 64),
    dropout=0.10,
    learn_alpha=False,           # using external schedule (can set True to learn per-block)
    alpha_init=0.0,
    add_aux=False
).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=5e-4)


In [9]:
def cosine_ramp(epoch, T):
    # returns 0→1 as epoch goes 0→T; clamp afterward
    t = max(0.0, min(float(epoch)/float(T), 1.0))
    return 0.5 - 0.5*math.cos(math.pi * t)

alpha_ramp_epochs = 60  # by ~60 epochs, behave like MLP


In [ ]:
epochs = 100
print_every = 1

model.train()
for epoch in range(1, epochs+1):
    total_loss = 0.0
    correct = 0
    total = 0
    alpha_t = torch.tensor(cosine_ramp(epoch, alpha_ramp_epochs), device=device)

    for batch in loader:
        batch = batch.to(device)
        optimizer.zero_grad()

        out = model(batch.x, batch.edge_index, alpha_override=alpha_t, return_all=True)
        logits = out["logits"]
        z3 = out["z3"]

        # main CE loss
        loss = F.cross_entropy(logits, batch.y)

        # mild variance/whitening regularization on z3 to avoid collapse
        zc = z3 - z3.mean(0, keepdim=True)
        cov = (zc.T @ zc) / (max(1, zc.shape[0]-1))
        var_loss = -torch.log(torch.clamp(torch.diag(cov), min=1e-6)).mean() \
                   + 0.05 * (cov - torch.diag(torch.diag(cov))).pow(2).mean()
        loss = loss + 0.05 * var_loss

        loss.backward()
        optimizer.step()
        total_loss += float(loss)

        # --- Accuracy computation ---
        preds = logits.argmax(dim=1)        # class with highest logit
        correct += (preds == batch.y).sum().item()
        total += batch.y.size(0)

    acc = correct / total
    print(f"Epoch {epoch}, Loss: {total_loss:.4f}, Accuracy: {acc:.4f}")

Epoch 1, Loss: 44.8357, Accuracy: 0.0835
Epoch 2, Loss: 43.1178, Accuracy: 0.1478
Epoch 3, Loss: 40.4184, Accuracy: 0.1482
Epoch 4, Loss: 38.2124, Accuracy: 0.1656
Epoch 5, Loss: 37.9094, Accuracy: 0.1959
Epoch 6, Loss: 37.7413, Accuracy: 0.1965


In [ ]:
model.eval()
rows = []
with torch.no_grad():
    for data in dataset:
        data = data.to(device)
        out = model(data.x, data.edge_index, alpha_override=torch.tensor(1.0, device=device), return_all=True)
        z3 = out["z3"].cpu().numpy()
        logits = out["logits"].cpu().numpy()
        preds = logits.argmax(axis=1)

        x_pos = data.x_pos.cpu().numpy()
        y_pos = data.y_pos.cpu().numpy()
        region_id = data.region_id
        labels_num = data.y.cpu().numpy()
        labels_str = label_encoder.inverse_transform(labels_num)

        for i in range(z3.shape[0]):
            rows.append({
                "region": region_id,
                "x_pos": float(x_pos[i]),
                "y_pos": float(y_pos[i]),
                "true_label": labels_str[i],
                "pred_label": label_encoder.inverse_transform([preds[i]])[0],
                "z1": float(z3[i,0]),
                "z2": float(z3[i,1]),
                "z3": float(z3[i,2])
            })

embed_df = pd.DataFrame(rows)
embed_df.head(), embed_df.shape


In [ ]:
z_vals = embed_df[["z1","z2","z3"]].values
z_min = z_vals.min(axis=0)
z_max = z_vals.max(axis=0)
rng = np.clip(z_max - z_min, 1e-9, None)
rgb = ((z_vals - z_min) / rng * 255.0).astype(np.uint8)

embed_df["R"] = rgb[:,0]
embed_df["G"] = rgb[:,1]
embed_df["B"] = rgb[:,2]

out_csv = "node_embeddings_3d_ce.csv"
embed_df.to_csv(out_csv, index=False)
print("Saved:", out_csv, " | rows:", len(embed_df))


In [ ]:
import cv2

save_dir = "region_images_ce"
os.makedirs(save_dir, exist_ok=True)

for reg, sub in embed_df.groupby("region"):
    xs = sub["x_pos"].values
    ys = sub["y_pos"].values
    Rs = sub["R"].values
    Gs = sub["G"].values
    Bs = sub["B"].values

    H, W = 1024, 1024  # adjust to your canvas
    img = np.zeros((H, W, 3), dtype=np.uint8)

    for i in range(len(sub)):
        x = int(round(xs[i]))
        y = int(round(ys[i]))
        if 0 <= x < W and 0 <= y < H:
            img[y, x, 0] = Bs[i]   # OpenCV uses BGR
            img[y, x, 1] = Gs[i]
            img[y, x, 2] = Rs[i]

    path = os.path.join(save_dir, f"region_{reg}.png")
    cv2.imwrite(path, img)
    print("Saved", path)


In [ ]:
torch.save(model.state_dict(), "gcn3d_ce_model.pth")
print("Saved weights: gcn3d_ce_model.pth")


## Model Evaluation

Now that the model has been trained, we can evaluate its performance on a separate test set. This will provide an unbiased assessment of how well the model generalizes to unseen data.

In [ ]:
# Assuming a test set split is needed.
# If you have a separate file for the test set, load it here.
# For this example, let's assume a simple split of the existing dataset for demonstration.
# In a real scenario, ensure proper splitting based on your data structure (e.g., by region).

# Example: Splitting the dataset (replace with your actual test set loading/creation)
# from torch_geometric.data import DataLoader
# train_size = int(0.8 * len(dataset))
# test_size = len(dataset) - train_size
# train_dataset, test_dataset = torch.utils.data.random_split(dataset, [train_size, test_size])

# test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

# For now, let's use the full dataset as a placeholder for evaluation if no explicit test set is defined
# IMPORTANT: Replace this with your actual test data loading and processing
test_loader = DataLoader(dataset, batch_size=batch_size, shuffle=False)

print(f"Number of test batches: {len(test_loader)}")